# E1 · CLIP ViT-B/32 sobre la imagen completa (línea base)

**Pregunta.** ¿Un encoder visual preentrenado, sin ningún ajuste, ya ordena correctamente a la mascota buscada entre las de su especie?

| | |
|---|---|
| Pipeline | imagen completa → CLIP ViT-B/32 (OpenAI, revisión fija `3d74acf9`) → vector de 512 dimensiones, norma L2 |
| Entrenamiento | ninguno (*zero-shot*) |
| Modelo | `pettrace-embedder` versión 1 (`clip-vit-b32@v1`), en producción en la app |
| Dataset | `pettrace-reid@v3` (privado en Hugging Face; animales con códigos seudónimos, sin fotos en este notebook) |
| Protocolo | cada foto es una consulta; galería = las demás fotos de la misma especie (*leave-one-out*); cada animal puntúa con su mejor foto; Rank-1/5/10, mAP, EER |

Reproducir: `make image && make notebook NB=notebooks/e1_clip_zero_shot.ipynb` (ver el README). El notebook registra una ejecución en MLflow con el commit del código, el commit del dataset y el sha256 del modelo, y adjunta figuras, tablas y este mismo notebook.

In [ ]:
import json, os

# Paths are relative to the repository root (nbconvert starts the kernel in notebooks/).
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
os.environ.setdefault("TQDM_DISABLE", "1")
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
from collections import Counter

from IPython.display import Image, Markdown, display

from reid import data
from reid.evaluation import evaluate, load_model, log_run, report
from reid.figures import metrics_table, save

print("código  :", os.environ.get("GIT_SHA", "?")[:7], "(sin cambios locales)" if os.environ.get("GIT_DIRTY") == "false" else "(¡cambios sin commit!)")
print("MLflow  :", os.environ.get("MLFLOW_TRACKING_URI"))

EXPERIMENT, DATASET, MODEL, REPORT = "E1", "v3", "models:/pettrace-embedder/1", "reports/e1_v3"

## 1. Datos

In [ ]:
ds = data.load(DATASET)
bucket = lambda x: "oscura" if x < 0.25 else "tenue" if x < 0.4 else "normal"
rows = ["| Especie | Animales | Fotos | Fotos por animal | Luz normal / tenue / oscura | Particiones |", "|---|---|---|---|---|---|"]
for sp in ("cat", "dog"):
    ph = [p for p in ds.photos if p.species == sp]
    per = Counter(p.animal for p in ph)
    light = Counter(bucket(p.light) for p in ph)
    splits = Counter(ds.animals[a]["split"] for a in per)
    rows.append(f"| {sp} | {len(per)} | {len(ph)} | {min(per.values())}–{max(per.values())} | "
                f"{light['normal']} / {light['tenue']} / {light['oscura']} | {dict(splits)} |")
display(Markdown(f"**{ds.name}** · commit `{ds.commit}` · huella `{ds.fingerprint}`\n\n" + "\n".join(rows)))
groups = Counter(r["lookalike_group"] for r in ds.animals.values() if r["lookalike_group"])
display(Markdown("Grupos de parecido (negativos difíciles): " + ", ".join(
    f"`{g}`: " + ", ".join(a for a, r in sorted(ds.animals.items()) if r["lookalike_group"] == g) for g in sorted(groups))))

## 2. Modelo

Se descarga la versión registrada y se verifica que es el modelo que dice ser: el *golden set* (12 imágenes generadas) re-embebido con su propio pipeline debe coincidir con los embeddings guardados al registrarla.

In [ ]:
model = load_model(MODEL)
display(Markdown("\n".join([
    "| | |", "|---|---|",
    f"| URI | `{model.uri}` (versión {model.registry_version} del registro) |",
    f"| Pipeline | `{model.model_version}` |",
    *[f"| sha256 `{f}` | `{h}` |" for f, h in model.sha256.items()],
    f"| Golden set | huella `{model.golden_fingerprint}`, {'✓ reproduce los embeddings guardados' if model.golden_ok else '✗ NO coincide'} |",
])))
assert model.golden_ok, "the downloaded model does not reproduce its golden fingerprint"

## 3. Entrenamiento

No aplica: E1 es la línea base *zero-shot*. El encoder se usa tal como fue preentrenado.

## 4. Evaluación

In [ ]:
ev = evaluate(ds, model.pipeline, zero_shot=True)
print(report(ev.result))

## 5. Resultados

In [ ]:
s1 = save(ev, model, REPORT)
display(Markdown(metrics_table({"E1": s1})))

In [ ]:
for name in ("confusion", "rank1_por_animal", "distribucion_scores", "cmc", "matriz_similitud"):
    display(Image(os.path.join(REPORT, f"{name}.png"), width=760))

## 6. Registro en MLflow

In [ ]:
run_id = log_run(ev, model, experiment=EXPERIMENT, artifacts=REPORT)
json.dump({"run_id": run_id, "experiment": EXPERIMENT, "dataset": ds.name, "dataset_commit": ds.commit,
           "model": MODEL, "git_sha": os.environ.get("GIT_SHA", "unknown")}, open(os.path.join(REPORT, "run.json"), "w"), indent=2)
display(Markdown(f"Ejecución MLflow: [`{run_id[:8]}`]({os.environ['MLFLOW_TRACKING_URI']}/#/experiments/"
                 f"{__import__('mlflow').get_experiment_by_name('pettrace-reid-eval').experiment_id}/runs/{run_id})"))

## 7. Conclusiones

- **El ranking funciona muy por encima del azar**: el animal correcto queda primero en ~73 % de las consultas frente a 14 % por azar, y casi siempre está entre los 5 primeros. Es la base de la decisión de producto: mostrar un *ranking* que el dueño confirma.
- **Un umbral fijo no separa identidades**: las similitudes de pares de animales distintos llegan a valores tan altos como las del mismo animal (EER ≈ 39 %). No se usan alertas automáticas por umbral.
- **Los errores se concentran en los grupos de parecido** (atigrados, blancos de pelo rizado), lo que motiva E2 (quitar el fondo) y E3–E5 (especializar con el dataset propio y con la descripción).
- Muestra pequeña (14 animales, 128 consultas): los resultados se leen como tendencia; los intervalos de confianza están pendientes (ver `TODO.md`).